[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/python_binding_examples/lsdtt3_bindings_quickstart.ipynb)

# lsdtt3 Python bindings quickstart: the Lauterbrunnen valley

The other notebooks in this repository run the `lsdtt3` command-line programs with parameter files. This one uses the **Python bindings** instead: every step is an ordinary Python function call, and the rasters stay in memory as `lsdtt3.Raster` objects.

In this notebook you will:

1. Install the `lsdtt3` Python wheel.
2. Download a DEM of the Lauterbrunnen valley in the Bernese Alps, Switzerland, with `lsdtt3.fetch_dem`.
3. Route flow, extract the channel network and plot it over a hillshade.
4. Compute a slope (topographic gradient) map and plot it.

We plot with `matplotlib` here to keep the dependencies small. For nicer maps, the LSDTopoTools visualisation package is `lsdviztools3`.

## Install lsdtt3

The wheel is built for **Linux x86_64 and CPython 3.13 only**. Colab runs Linux, but check the Python version first: if it is not 3.13, `pip` will refuse to install the wheel.

In [ ]:
!python --version

Now install the wheel. It bundles everything `lsdtt3` needs (including GDAL and PROJ), so no conda setup is required.

In [ ]:
!pip install https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdtt3-0.7.1-cp313-cp313-manylinux_2_28_x86_64.whl

Import the package and check the versions. You should see `0.7.1` for `lsdtt3` and `3.10.3` for the GDAL library bundled inside it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import lsdtt3

print("lsdtt3 version:", lsdtt3.__version__)
print("GDAL version:  ", lsdtt3._core.gdal_version())

## Download a DEM

`lsdtt3.fetch_dem` downloads a DEM for a longitude/latitude bounding box, mosaics the tiles and returns an in-memory `Raster`. The default source, `cop30_aws`, is the Copernicus GLO-30 DEM, which needs **no API key**.

The Copernicus tiles are in geographic coordinates (degrees), but topographic analysis needs square pixels measured in metres. `fetch_dem` therefore reprojects the DEM for you: with the default `target_epsg=0` it picks the UTM zone at the centre of the bounding box (here UTM zone 32N, EPSG:32632), and `grid_spacing` sets the pixel size in metres.

Our box covers the Lauterbrunnen valley, from Lauterbrunnen village south to the head of the valley below the Breithorn, roughly 10 km by 11 km.

In [ ]:
# Bounding box in decimal degrees (WGS84)
west = 7.84
east = 7.97
south = 46.50
north = 46.60

dem = lsdtt3.fetch_dem(
    west=west,
    south=south,
    east=east,
    north=north,
    source="cop30_aws",
    grid_spacing=30.0,
)

metadata = dem.metadata
print("Rows, columns:", dem.shape)
print("Pixel size (m):", metadata.cell_size)

# The coordinate reference system is stored as WKT text; its name is the first quoted string
crs_name = metadata.crs_wkt.split('"')[1]
print("Coordinate reference system:", crs_name)
print("UTM EPSG code for the box centre:", lsdtt3.utm_epsg_for_point((west + east) / 2, (south + north) / 2))

It is a good idea to save the DEM so you can open it later (for example in QGIS). `write` produces a Cloud Optimised GeoTIFF by default.

In [ ]:
dem.write("lauterbrunnen_dem.tif")

`lsdtt3` rasters carry a validity mask alongside the values. This small helper turns a `Raster` into a NumPy array with `NaN` where there is no data, which is what `matplotlib` wants. We also work out the map extent in metres so the axes show UTM coordinates.

In [ ]:
def raster_to_array(raster):
    values = raster.to_numpy()
    valid = raster.mask_numpy()
    return np.where(valid, values, np.nan)


map_extent = [metadata.x_min, metadata.x_max, metadata.y_min, metadata.y_max]

## Extract the channel network

Channel extraction takes three steps:

1. **Fill depressions** with `lsdtt3.fill_pits`, so that every pixel can drain to the edge of the DEM. A small minimum slope is imposed across filled areas so they are not perfectly flat.
2. **Route flow** with `lsdtt3.FlowInfo`, which builds the D8 (steepest descent) drainage topology.
3. **Pick the channels**: `channel_scope` selects channel pixels whose contributing area exceeds a threshold number of pixels, and `channel_network` organises them into links and junctions with Strahler stream orders.

With 30 m pixels, a threshold of 200 pixels means a drainage area of about 0.18 km². Try changing it to see how the network grows or shrinks.

In [ ]:
filled_dem = lsdtt3.fill_pits(dem)

flow = lsdtt3.FlowInfo(filled_dem)

threshold_contributing_pixels = 200
scope = flow.channel_scope(threshold_contributing_pixels)
network = flow.channel_network(scope)

print("Number of channel pixels:", network.n_channel_cells)
print("Number of junctions:", network.n_junctions)

Now make a hillshade with `lsdtt3.hillshade` and plot the channels on top. `network.lines_geodataframe()` returns the channel links as a GeoPandas table of lines in the DEM's UTM coordinates, with a `stream_order` column (Strahler order), so we can colour each channel by its order.

In [ ]:
hillshade = lsdtt3.hillshade(dem)
hillshade_array = raster_to_array(hillshade)

channel_lines = network.lines_geodataframe()
print(channel_lines.head())

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(hillshade_array, cmap="gray", extent=map_extent)
channel_lines.plot(
    ax=ax,
    column="stream_order",
    cmap="winter_r",
    linewidth=1.0,
    legend=True,
    legend_kwds={"label": "Strahler stream order", "shrink": 0.7},
)
ax.set_title("Lauterbrunnen channel network")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

## Compute a slope map

`lsdtt3.polyfit_metrics` fits a quadratic surface to a moving window around each pixel and computes surface metrics from the fit. The window size is set by `lambda_metres`; a value of a few pixels (here 90 m) smooths out pixel-scale noise in the DEM.

We ask for `"slope"`, which is the **topographic gradient**: rise over run, in metres per metre (dimensionless). A gradient of 1 means the surface rises one metre for every metre travelled horizontally; the cliffs on either side of the Lauterbrunnen valley are steeper than that.

In [ ]:
surface_metrics = lsdtt3.polyfit_metrics(dem, ["slope"], lambda_metres=90.0)
slope = surface_metrics["slope"]
slope_array = raster_to_array(slope)

print("Median slope (m/m):", np.nanmedian(slope_array))

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(hillshade_array, cmap="gray", extent=map_extent)
slope_image = ax.imshow(slope_array, cmap="viridis", extent=map_extent, alpha=0.6, vmin=0, vmax=2)
fig.colorbar(slope_image, ax=ax, shrink=0.7, label="Slope (m/m)")
ax.set_title("Lauterbrunnen topographic gradient")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

If you want to keep the slope raster, write it out:

In [ ]:
slope.write("lauterbrunnen_slope.tif")

## Where next

The bindings expose much more than this. To explore them:

* `help(lsdtt3.FlowInfo)` lists the flow-routing methods (drainage area, basins, chi, height above nearest drainage, and so on).
* `help(lsdtt3.ChannelNetwork)` lists what you can do with the channel network (junction and link tables, chi, link profiles).